In [1]:
import requests
from datetime import datetime
import time
import os
import pandas as pd

class KOMISCrawler:
    def __init__(self):
        self.base_url = "https://www.komis.or.kr"
        self.session = requests.Session()
        self.session.headers.update({
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
            'Accept': 'application/json, text/javascript, */*; q=0.01',
            'Accept-Language': 'ko-KR,ko;q=0.9',
            'Content-Type': 'application/x-www-form-urlencoded; charset=UTF-8',
            'X-Requested-With': 'XMLHttpRequest',
            'Origin': 'https://www.komis.or.kr'
        })

        # 각 카테고리별로 실제 웹사이트에서 선택 가능한 광종만 포함
        self.categories = {
            'HP001': {  # 비철금속
                'name': '비철금속',
                'url': '/Komis/RsrcPrice/BaseMetals',
                'minerals': {
                    '니켈': 'MNRL0002',
                    '동': 'MNRL0008',
                    '아연': 'MNRL0023',
                    '알루미늄': 'MNRL0009',
                    '연': 'MNRL0022',
                    '주석': 'MNRL0016'
                }
            },
            'HP002': {  # 희소금속
                'name': '희소금속',
                'url': '/Komis/RsrcPrice/MinorMetals',
                'minerals': {
                    '리튬': 'MNRL0001',
                    '코발트': 'MNRL0003',
                    '망간': 'MNRL0004',
                    '크롬': 'MNRL0021',
                    '몰리브덴': 'MNRL0012',
                    '바나듐': 'MNRL0013',
                    '텅스텐': 'MNRL0018',
                    '티타늄': 'MNRL0017',
                    '안티모니': 'MNRL0019',
                    '니오븀': 'MNRL0007',
                    '규소': 'MNRL0010',
                    '마그네슘': 'MNRL0011',
                    '흑연': 'MNRL0005',
                    '갈륨': 'MNRL0024',
                    '게르마늄': 'MNRL0035',
                    '인듐': 'MNRL0025',
                    '셀레늄': 'MNRL0029',
                    '탄탈륨': 'MNRL0026',
                    '지르코늄': 'MNRL0027',
                    '스트론튬': 'MNRL0028'
                }
            },
            'HP003': {  # 철광석 및 에너지
                'name': '철광석및에너지',
                'url': '/Komis/RsrcPrice/IronOre',
                'minerals': {
                    # '철광석': 'MNRL0XXX',  # 실제 웹사이트에서 확인 필요
                    '유연탄': 'MNRL0032'
                }
            },
            'HP004': {  # 기타 광석
                'name': '기타광석',
                'url': '/Komis/RsrcPrice/EtcMnrl',
                'minerals': {
                    '금': 'MNRL0046',
                    '은': 'MNRL0047',
                    '백금': 'MNRL0014',
                    '팔라듐': 'MNRL0015',
                    '우라늄': 'MNRL0031'
                }
            }
        }

    def update_referer(self, category_url):
        """카테고리별 Referer 헤더 업데이트"""
        self.session.headers['Referer'] = self.base_url + category_url

    def get_price_criteria(self, mineral_code, hp000='HP001'):
        """광종별 가격기준 조회"""
        url = f"{self.base_url}/Komis/RsrcPrice/ajax/getMnrlPriceCrtr"

        data = {
            'HP000': hp000,
            'mnrkndUnqCd': mineral_code
        }

        try:
            response = self.session.post(url, data=data, timeout=10)
            if response.status_code == 200:
                result = response.json()
                if result.get('data') and len(result['data']) > 0:
                    return result['data'][0].get('cdKey', '')
        except Exception as e:
            print(f"가격기준 조회 오류: {e}")

        return ''

    def get_price_data(self, mineral_name, start_year=2024, end_year=2026,
                      avg_option='DAY', hp000='HP001'):
        """광종별 가격 데이터 조회"""

        if hp000 not in self.categories:
            print(f"지원하지 않는 카테고리: {hp000}")
            return None

        category = self.categories[hp000]
        minerals = category['minerals']

        if mineral_name not in minerals:
            print(f"지원하지 않는 광종: {mineral_name}")
            print(f"사용 가능한 광종: {list(minerals.keys())}")
            return None

        mineral_code = minerals[mineral_name]

        # Referer 업데이트
        self.update_referer(category['url'])

        price_criteria = self.get_price_criteria(mineral_code, hp000)

        print(f"{mineral_name} 데이터 조회 중...")
        print(f"  - 광종코드: {mineral_code}")
        print(f"  - 가격기준: {price_criteria}")

        # 데이터 조회 URL
        data_url = f"{self.base_url}/Komis/RsrcPrice/ajax/getMnrlPrcByMnrkndUnqCd"

        params = {
            'HP000': hp000,
            'srchMnrkndUnqCd': mineral_code,
            'srchPrcCrtr': price_criteria,
            'srchAvgOpt': avg_option,
            'srchField': 'year',
            'srchStartDate': str(start_year),
            'srchEndDate': str(end_year),
            'srchCompareMnrkndUnqCd': '',
            'srchComparePrcCrtr': '',
            'lmeInvt': 'N'
        }

        try:
            response = self.session.post(data_url, data=params, timeout=30)

            if response.status_code == 200:
                result = response.json()

                # 반환된 광종명 확인
                returned_mineral = result.get('dataAvg', {}).get('INFO', {}).get('mnrkndKornNm', '')
                print(f"  - 반환된 광종: {returned_mineral}")

                return result
            else:
                print(f"데이터 조회 실패: HTTP {response.status_code}")
                return None

        except Exception as e:
            print(f"데이터 조회 오류: {e}")
            return None

    def download_excel(self, mineral_name, start_year=2024, end_year=2026,
                      avg_option='DAY', hp000='HP001', output_dir=None):
        """엑셀 다운로드"""

        if output_dir is None:
            output_dir = r"C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\KOMIS"

        os.makedirs(output_dir, exist_ok=True)

        # 데이터 조회
        data = self.get_price_data(mineral_name, start_year, end_year, avg_option, hp000)

        if data and data.get('data'):
            try:
                default_data = data['data'].get('defaultMnrl', [])

                if default_data:
                    # 데이터프레임 생성
                    df = pd.DataFrame(default_data)

                    # 필요한 컬럼만 선택
                    columns_map = {
                        'crtrYmd': '기준일',
                        'cmercPrc': '기준가격',
                        'flctnPrc': '등락가',
                        'flctnPrcnt': '등락율(%)'
                    }

                    df_export = df[list(columns_map.keys())].copy()
                    df_export.columns = list(columns_map.values())

                    # 날짜 포맷 변경
                    if avg_option in ['DAY', 'WEEK']:
                        df_export['기준일'] = pd.to_datetime(df_export['기준일'], format='%Y%m%d').dt.strftime('%Y-%m-%d')
                    elif avg_option == 'MONTH':
                        df_export['기준일'] = df_export['기준일'].apply(lambda x: f"{str(x)[:4]}-{str(x)[4:]}")

                    # 엑셀 저장
                    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
                    category_name = self.categories[hp000]['name']
                    filename = f"{category_name}_{mineral_name}_{avg_option}_{start_year}_{end_year}_{timestamp}.xlsx"
                    filepath = os.path.join(output_dir, filename)

                    df_export.to_excel(filepath, index=False, engine='openpyxl')

                    print(f"✓ {mineral_name} 다운로드 완료: {filepath}")
                    print(f"  - 데이터 건수: {len(df_export)}건")
                    return True
                else:
                    print(f"✗ {mineral_name}: 데이터 없음")
                    return False

            except Exception as e:
                print(f"✗ {mineral_name} 엑셀 변환 오류: {e}")
                import traceback
                traceback.print_exc()
                return False
        else:
            print(f"✗ {mineral_name}: 데이터 조회 실패")
            return False

    def download_category(self, hp000='HP001', start_year=2024, end_year=2026,
                         avg_option='DAY', output_dir=None, mineral_list=None):
        """카테고리별 모든 광종 또는 특정 광종만 다운로드"""

        if output_dir is None:
            output_dir = r"C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\KOMIS"

        if hp000 not in self.categories:
            print(f"지원하지 않는 카테고리: {hp000}")
            return {}

        category = self.categories[hp000]
        category_name = category['name']
        minerals = category['minerals']

        # 특정 광종만 다운로드
        if mineral_list:
            minerals = {k: v for k, v in minerals.items() if k in mineral_list}
            if not minerals:
                print(f"선택한 광종이 {category_name}에 없습니다.")
                return {}

        print(f"\n{'='*60}")
        print(f"KOMIS {category_name} 가격 데이터 다운로드 시작")
        print(f"저장 경로: {output_dir}")
        print(f"기간: {start_year} ~ {end_year}")
        print(f"평균옵션: {avg_option}")
        print(f"광종 개수: {len(minerals)}개")
        print(f"{'='*60}\n")

        results = {}

        for mineral_name in minerals.keys():
            success = self.download_excel(
                mineral_name=mineral_name,
                start_year=start_year,
                end_year=end_year,
                avg_option=avg_option,
                hp000=hp000,
                output_dir=output_dir
            )
            results[mineral_name] = success
            print()
            time.sleep(1)  # API 부하 방지

        print(f"\n{'='*60}")
        print(f"{category_name} 다운로드 결과 요약")
        print(f"{'='*60}")
        success_count = sum(1 for v in results.values() if v)
        total_count = len(results)

        for mineral, success in results.items():
            status = "✓ 성공" if success else "✗ 실패"
            print(f"{mineral}: {status}")

        print(f"{'='*60}")
        print(f"성공: {success_count}/{total_count}")
        print(f"{'='*60}\n")

        return results

    def download_all_categories(self, start_year=2024, end_year=2026,
                                avg_option='DAY', output_dir=None):
        """모든 카테고리의 모든 광종 다운로드"""

        if output_dir is None:
            output_dir = r"C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\KOMIS"

        print("\n" + "="*80)
        print("KOMIS 전체 광종 가격 데이터 다운로드 시작")
        print("="*80)

        all_results = {}

        for hp000 in ['HP001', 'HP002', 'HP003', 'HP004']:
            category_avg_option = avg_option

            # 철광석및에너지와 기타광석의 일부는 DAY 미지원
            if hp000 == 'HP003':
                category_avg_option = 'WEEK'  # 유연탄은 WEEK 사용
            elif hp000 == 'HP004' and avg_option == 'DAY':
                # 우라늄만 DAY 미지원, 나머지는 DAY 가능
                pass

            results = self.download_category(
                hp000=hp000,
                start_year=start_year,
                end_year=end_year,
                avg_option=category_avg_option,
                output_dir=output_dir
            )
            all_results[hp000] = results

            time.sleep(2)  # 카테고리 간 대기

        # 전체 요약
        print("\n" + "="*80)
        print("전체 다운로드 결과 요약")
        print("="*80)

        total_success = 0
        total_count = 0

        for hp000, results in all_results.items():
            category_name = self.categories[hp000]['name']
            success = sum(1 for v in results.values() if v)
            count = len(results)
            total_success += success
            total_count += count
            print(f"{category_name}: {success}/{count} 성공")

        print(f"\n전체: {total_success}/{total_count} 성공")
        print("="*80 + "\n")

        return all_results

def main():
    output_path = r"C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\KOMIS"

    crawler = KOMISCrawler()

    # 사용 예시 1: 전체 다운로드
    crawler.download_all_categories(
        start_year=2024,
        end_year=2026,
        avg_option='DAY',
        output_dir=output_path
    )

    # # 사용 예시 2: 특정 카테고리만
    # crawler.download_category(
    #     hp000='HP001',  # 비철금속
    #     start_year=2024,
    #     end_year=2026,
    #     avg_option='DAY',
    #     output_dir=output_path
    # )

    # # 사용 예시 3: 특정 광종만
    # crawler.download_category(
    #     hp000='HP002',  # 희소금속
    #     start_year=2024,
    #     end_year=2026,
    #     avg_option='DAY',
    #     mineral_list=['리튬', '코발트'],  # 리튬과 코발트만
    #     output_dir=output_path
    # )

if __name__ == "__main__":
    main()


KOMIS 전체 광종 가격 데이터 다운로드 시작

KOMIS 비철금속 가격 데이터 다운로드 시작
저장 경로: C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\KOMIS
기간: 2024 ~ 2026
평균옵션: DAY
광종 개수: 6개



PermissionError: [WinError 5] 액세스가 거부되었습니다: 'C:\\Users\\82108'